[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/03_tools_and_schemas.ipynb)


# Agentic Systems Foundations
## Notebook 03: Tools and Schemas — Reach, and Reliability
**Duration:** 40 min &nbsp;|&nbsp; **Mode:** Guided Coding

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** the **ACT** step. The loop can think and remember; now it touches the real world.

> **Requires `OPENAI_API_KEY`.** These notebooks call a real model —
> there is no simulated fallback, on purpose.


In [ ]:
# ============================================================
# BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Works in Colab, a local venv, or a bare Jupyter. It installs whatever is
# actually MISSING rather than assuming a particular environment — checking by
# import is the only reliable test.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork

# import name -> pip package name
REQUIRED = {
    "openai": "openai",
    "dotenv": "python-dotenv",
    "jsonschema": "jsonschema",
    "langchain_core": "langchain-core",
    "langchain_openai": "langchain-openai",
    "langgraph": "langgraph",
}

def _present(module: str) -> bool:
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError, ModuleNotFoundError):
        return False

missing = sorted({pkg for mod, pkg in REQUIRED.items() if not _present(mod)})
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("dependencies: all present")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# YOUR API KEY  (required — there is no offline fallback)
# ============================================================
# Every notebook in this session calls a REAL model. There is deliberately no
# simulated fallback: a fake model can show you the shape of an agent loop, but
# it cannot show you how a real one behaves when your tool descriptions are
# ambiguous or your schema is too loose — and that behaviour is the subject of
# the session.
#
#   Colab : sidebar -> key icon -> add a secret named OPENAI_API_KEY
#           -> toggle "Notebook access" ON -> re-run this cell
#   local : export OPENAI_API_KEY=sk-...   (or put it in a .env file)
import os

def _load_key() -> bool:
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

if not _load_key():
    raise RuntimeError(
        "OPENAI_API_KEY is not set — this notebook calls a real model.\n"
        "Colab: sidebar -> key icon -> add OPENAI_API_KEY -> Notebook access ON.\n"
        "Local: export OPENAI_API_KEY=sk-...  then restart the kernel."
    )

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
print()
print("These notebooks spend real tokens. Budgets are deliberately small.")

## WHY — the schema is where you decide how much to trust the model

A tool is a promise: *call me with these arguments and I will do this thing.*
An LLM is a text generator that has read a description of that promise and is
doing its best. Every tool-calling bug you will ever debug lives in that gap.

Make it concrete. One tool, `get_order_status(order_id)`. One model output:

```json
{"order_id": 1042}
```

An integer, not `"ACME-1042"`. What happens next is **entirely** determined by
your schema:

| Schema | What happens | What you see |
|---|---|---|
| **None / loose** | `1042` reaches your function; `orders[1042]` raises | a `KeyError` traceback blaming your *database code* for a mistake the model made two layers up |
| **Strict** | rejected before your function is entered | `order_id: 1042 does not match ^ACME-\d{4}$` — an error the agent can act on next step |

Same model, same output, completely different day. That is why schema design is
a first-class engineering skill and not paperwork.

> **The recurring question, for this step:** a schema failure shows up in the
> trace as a `REJECTED` observation. If you see the *same* parameter rejected
> twice, your error message is not telling the model what it needs to know.


## WHAT — anatomy of a good tool

Four parts, in descending order of how often people get them wrong:

1. **The description** — says *when* to use it, not just what it does. This is
   what the model reads to choose. Most "the model picked the wrong tool" bugs
   are actually "two descriptions did not say how they differ".
2. **Parameter descriptions with formats and examples** — the only natural-language
   guidance the model gets about what a valid value looks like.
3. **Constraints** — `enum` and `pattern`. An `enum` converts "guess a valid
   value" into "pick from this list". It is the highest-value line in any schema.
4. **The return value** — *prose written for a model to read*, not a status code.

And one rule that is not optional:

> ### Tools must never raise
> A tool that raises kills the loop and takes the trace with it. A tool that
> *returns* a readable error lets the agent read it, fix its arguments and
> recover on the next step. Same failure; one ends the demo, the other becomes
> a self-repair you can point at.


## HOW (from scratch) — a schema, derived from the function itself

In [ ]:
# We DERIVE the schema from the Python signature + type hints + docstring.
# Never hand-write a schema next to a function: that is a second source of
# truth, and it rots the first time someone adds a parameter.
import json
from agent_core.acme_tools import acme_registry

tools = acme_registry()
refund = tools.get("check_refund_eligibility")

print("DESCRIPTION (what the model reads to CHOOSE this tool):")
print(" ", refund.description[:200], "...\n")
print("SCHEMA (what the model must SATISFY to call it):")
print(json.dumps({k: v for k, v in refund.schema.items() if not k.startswith("_")}, indent=2))

Look at what the docstring bought us:

- `reason: Literal["billing_error", ...]` became an **`enum`**. The model is now
  picking from a list rather than inventing a plausible string.
- `pattern: ^ACME-\d{4}$` in the docstring became a real **regex constraint**.
- `additionalProperties: false` means an invented parameter is *rejected*, not
  silently passed through. Most tutorials leave this off; every system that has
  been burned turns it on.


> ### ✋ Predict before you run
> We are about to send four deliberately wrong payloads to these tools: a bare number where an `ACME-####` ID is required, a made-up reason code, an invented extra parameter, and the string `"3"` where an integer is expected. **Which of the four should be rejected, and which should be quietly fixed?** Is 'reject everything invalid' the right answer?
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# Validation AND coercion. Note that these are different jobs.
from agent_core.schemas import validate_args

cases = [
    ("bare number as ID",   "get_order_status",        {"order_id": 1042}),
    ("invented reason",     "check_refund_eligibility", {"order_id": "ACME-1042", "reason": "vibes"}),
    ("invented parameter",  "get_order_status",        {"order_id": "ACME-1042", "urgent": True}),
    ('string "3" for int',  "search_docs",             {"query": "pricing", "k": "3"}),
]

for label, tool_name, payload in cases:
    schema = tools.get(tool_name).schema
    checked = validate_args(payload, schema)
    verdict = "OK " if checked.ok else "REJECTED"
    print(f"[{verdict}] {label}")
    print(f"          sent : {payload}")
    if checked.coercions:
        print(f"          FIXED: {checked.coercions}")
    if checked.errors:
        print(f"          why  : {checked.message()}")
    print()

**What you should observe:** the string `"3"` was **coerced** to the integer `3`
rather than rejected. That is deliberate, and it is the part stock validators do
not do for you.

`"3"` instead of `3` is a *model quirk*, not a user error. Rejecting it burns a
whole agent step — an LLM call, latency, money — to fix something unambiguous.
So we fix it and **record that we did**, which is what `coercions` is for. You
can see exactly what the validator quietly repaired.

The other three are genuinely ambiguous or genuinely wrong, so they are rejected
with a message written **for the model to read**. Compare:

- `order_id: 1042 does not match required format ^ACME-\d{4}$` ← actionable
- `ValidationError at $.order_id` ← the agent will guess

That difference is why a retry loop converges instead of spinning.


## HOW — the same schema, three provider interfaces

In [ ]:
# The agenda asks about "schema design across different LLM interfaces".
# Here is the honest summary: every provider accepts the SAME JSON Schema and
# disagrees only about the envelope around it.
import json
lookup = tools.get("get_order_status")

for label, payload in [
    ("OpenAI",    lookup.to_openai()),
    ("Gemini",    lookup.to_gemini()),
    ("Anthropic", lookup.to_anthropic()),
]:
    print(f"--- {label} " + "-" * (56 - len(label)))
    print(json.dumps(payload, indent=1)[:340])
    print()

**The point of that cell:** the envelope differs — `{"type":"function", "function":…}`
vs a bare declaration vs `input_schema` — and the *schema inside is identical*.

The envelope is trivia you look up in five minutes. The schema — the types, the
enum, the pattern, the descriptions — is the design work, and it **transfers
unchanged between providers**. A learner who internalises that will not be
thrown by the next SDK.

(One real wrinkle worth noticing: our Gemini adapter drops `additionalProperties`,
because Gemini's schema dialect has not supported it. That is the shape of
provider differences — small adapters, not redesigns.)


## HOW — tools must never raise

In [ ]:
# Four ways to abuse a tool. Not one of them raises.
abuses = [
    ("a tool that does not exist", "lookup_customer", {"id": "1"}),
    ("division by zero",           "calculate",       {"expression": "1/0"}),
    ("code injection attempt",     "calculate",       {"expression": "__import__('os').system('ls')"}),
    ("well-formed, unknown ID",    "get_order_status", {"order_id": "ACME-9999"}),
]

for label, name, args in abuses:
    obs = tools.dispatch(name, args)
    print(f"{label:<30} -> status={obs.status.value:<9} {obs.summary(70)}")

print("\nThe loop is still alive. Every failure became an OBSERVATION the agent can read.")

Notice the three distinct statuses, because the difference matters:

| Status | Meaning | Cost | Who is at fault |
|---|---|---|---|
| `REJECTED` | never ran — validation refused it | cheap | the model; fixable next step |
| `ERROR` | ran and blew up | expensive — side effects may have happened | could be either |
| `OK` | worked | — | — |

An agent that cannot tell `REJECTED` from `ERROR` cannot learn from either.

Note also `ACME-9999`: a **well-formed but unknown** ID. The schema cannot catch
that — it is a perfectly valid string. The *tool* catches it and says `NOT FOUND`
in words. If it had returned `None` or `{}`, the model would have treated the
silence as licence to invent an order. **Always say "not found" out loud.**


> ### ✋ Predict before you run
> Now the headline comparison. We give the agent an order ID written as `1046` instead of `ACME-1046`, once with the strict schema and once with a loosened copy that has no pattern constraint. **Which run ends better?** Careful — the intuitive answer is often wrong here.
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# STRICT vs LOOSE schema, same bad input.
import copy
from agent_core import Agent, Skill, ToolRegistry, compare
from agent_core.acme_tools import acme_registry

goal = "What is the status of order 1046?"   # note: missing the ACME- prefix

# --- build a LOOSE twin of get_order_status: pattern constraint removed -----
loose_tool = copy.deepcopy(acme_registry().get("get_order_status"))
loose_tool.schema["properties"]["order_id"].pop("pattern", None)

def make(tool):
    reg = ToolRegistry([tool])
    return Agent(skill=Skill(name="lookup",
                             instructions="Look up the order the user names.",
                             tools=reg))

strict = make(acme_registry().get("get_order_status")).run(goal)
loose  = make(loose_tool).run(goal)

print(compare({"STRICT schema": strict.trace, "LOOSE schema": loose.trace}))
print()
for label, run in (("STRICT", strict), ("LOOSE", loose)):
    obs = run.state.observations[0] if run.state.observations else None
    print(f"{label:<8} first observation -> {obs.status.value if obs else 'none'}: "
          f"{(obs.error or str(obs.result))[:96] if obs else ''}")

**What you should observe:** the strict schema **rejects** the call before the
tool runs, with a message naming the required format. The loose schema lets
`"1046"` through to the function, which then has to deal with it — and reports a
NOT FOUND for an ID that was never valid in the first place.

The loose version *looks* more forgiving and is actually worse: the failure has
moved from a cheap, precise, pre-execution rejection into your business logic,
where the error message can no longer tell the model what the right format was.

> **Constrain at the boundary.** Every check you skip in the schema becomes a
> check inside your function — written by someone with less context, reported
> in a message the model cannot act on.


## HOW — build your own tool (2 minutes, then we compare)

In [ ]:
# YOUR TURN. Write a tool the Acme agent does not have yet.
#
# Requirements — each maps to something above:
#   * a docstring summary that says WHEN to use it
#   * an Args: section describing every parameter
#   * a Literal[...] for any finite value set          -> becomes an enum
#   * `pattern: <regex>` in a description if a format matters
#   * return a STRING that reads like an answer, and never raise
from typing import Literal
from agent_core import tool

@tool(examples=["How many API calls are left on ACME-1042?"])
def check_usage(
    order_id: str,
    metric: Literal["api_calls", "storage_gb", "seats"] = "api_calls",
) -> str:
    """Report current usage against the plan limit for an order.

    Use this when a customer asks how much of their quota they have consumed,
    or whether they are near an overage charge.

    Args:
        order_id: The Acme order identifier, for example ACME-1042. pattern: ^ACME-\\d{4}$
        metric: Which usage figure to report.

    Returns:
        The current usage, the plan limit, and whether an overage applies.
    """
    # A stub with plausible numbers — the schema is the lesson here, not the data.
    figures = {"api_calls": (4_120_000, 5_000_000), "storage_gb": (712, 1024), "seats": (11, 15)}
    used, limit = figures[metric]
    pct = used / limit * 100
    flag = " — OVERAGE CHARGES APPLY" if pct > 100 else ""
    return f"Order {order_id}: {metric} at {used:,} of {limit:,} ({pct:.1f}% of plan limit){flag}."

import json
print(json.dumps({k: v for k, v in check_usage.schema.items() if not k.startswith("_")}, indent=1))
print("\nAs the model sees it in a prompt:")
print(check_usage.prompt_line())

In [ ]:
# Does it survive abuse? (It must, before it goes anywhere near an agent.)
from agent_core import ToolRegistry
reg = ToolRegistry([check_usage])
for args in ({"order_id": "ACME-1042"},
             {"order_id": "1042"},
             {"order_id": "ACME-1042", "metric": "bandwidth"}):
    obs = reg.dispatch("check_usage", args)
    print(f"{str(args):<58} -> {obs.status.value:<9} {obs.summary(60)}")

## HOW (parallel mapping) — `jsonschema` and LangChain

We hand-rolled a validator so nobody thinks validation is magic. Production
alternatives, and what they do and do not give you:


In [ ]:
# 1. jsonschema — the standards-compliant validator.
try:
    import jsonschema
    schema = {k: v for k, v in tools.get("get_order_status").schema.items()
              if not k.startswith("_")}
    try:
        jsonschema.validate({"order_id": 1042}, schema)
        print("jsonschema: accepted (unexpected!)")
    except jsonschema.ValidationError as e:
        print("jsonschema REJECTED:", e.message)
    print("\n-> Correct, and strictly stricter than us: it will NOT coerce \"3\" to 3.")
    print("   In an agent loop you usually want that coercion, so most teams end")
    print("   up writing the layer we wrote anyway — on top of a real validator.")
except ImportError:
    print("jsonschema not installed — concept still holds (see the notes above).")

In [ ]:
# 2. LangChain + Pydantic — how you would actually declare these schemas.
# No key needed for this cell: schemas are declarative.
import json
from agent_lc.tools_lc import ACME_TOOLS

lc_refund = next(t for t in ACME_TOOLS if t.name == "check_refund_eligibility")
schema = lc_refund.args_schema.model_json_schema()
print(json.dumps(schema["properties"], indent=1)[:900])

Compare that with the schema we derived from a docstring at the top of this
notebook. **Same contract, declared instead of inferred:**

| From scratch (`agent_core`) | Production (`agent_lc`) |
|---|---|
| `build_schema()` parsing signatures + docstrings | `args_schema=RefundArgs` (a Pydantic model) |
| `pattern:` marker in the docstring | `Field(pattern=r"^ACME-\d{4}$")` |
| `Literal[...]` → our `_json_type()` | `Literal[...]` → Pydantic (same!) |
| `validate_args()` — ~90 lines we wrote | Pydantic's validator |

**What the framework did for you:** the reflection, the validation, the JSON
Schema generation. Faster, more standards-complete, far better tested.

**What it did NOT do — and never will:**

- choose your enum values
- decide that an order ID looks like `ACME-####`
- write a description that says *when* to use the tool rather than what it does
- decide what your tool returns when it finds nothing

> Every one of those is a design decision, and every one of them is still yours.
> **The framework abstracts the mechanics, not the design decisions.**


In [ ]:
# One real difference worth knowing: what happens when a tool blows up.
#
# agent_core.Tool.invoke() NEVER raises — every failure becomes an Observation.
# LangGraph's ToolNode has the same behaviour behind a switch:
#
#     ToolNode(tools, handle_tool_errors=True)    # default: error -> ToolMessage
#     ToolNode(tools, handle_tool_errors=False)   # raises -> kills the graph run
#
# Same design decision we argued for, exposed as a flag. Leave it on.
from agent_lc.tools_lc import ACME_TOOLS
lookup = next(t for t in ACME_TOOLS if t.name == "get_order_status")

print("valid  :", lookup.invoke({"order_id": "ACME-1046"})[:64], "...")
try:
    lookup.invoke({"order_id": "1046"})
except Exception as e:
    print("invalid:", type(e).__name__, "- rejected by Pydantic before the function ran")
print()
print("-> Identical outcome to our validate_args() gate. Constrain at the boundary.")

## Recap

- The schema is where you choose how much to trust the model. **Constrain at the
  boundary**, before the call reaches your code.
- `enum` and `pattern` are the highest-value lines you can write. `Literal[...]`
  gives you an enum for free.
- **Derive** schemas from the function; never maintain them alongside it.
- Coerce what is unambiguous, reject what is not, and **write errors for the
  model to read**.
- **Tools must never raise.** Every failure becomes an observation instead.
- A tool that finds nothing must say so in words. Silence gets filled with
  invention.

**Next → Notebook 04 (Skills):** the agent can act reliably. Now — how does this
survive growing from five tools to twenty-five?
